# 04 · Feature Engineering, Machine Learning & Evaluation

Lecture 1 — From Messy Medical Data to Statistical Insight

*Created by Master of AI.*

> **Realistic training data, prepared for teaching.**

> **Teaching target — not a clinically validated prediction.**

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
DATA = "../data/"
df = pd.read_csv(DATA + "clean_patient_screening_data.csv", parse_dates=["visit_date"])
print(df.shape)
df.head()

## 1. Feature engineering

In [ ]:
df["is_smoker"] = (df["smoking_status"] == "Smoker").astype(int)
df["is_male"] = (df["sex"] == "Male").astype(int)
df["pulse_pressure"] = df["systolic_bp"] - df["diastolic_bp"]
df["bmi_category"] = pd.cut(df["bmi"], [0, 18.5, 25, 30, 200], right=False,
                            labels=["Underweight", "Normal", "Overweight", "Obese"])
df["bmi_category"].value_counts()

## 2. Features, target, split
`risk_score` and `follow_up_days` are excluded — they would **leak** the answer.

In [ ]:
from sklearn.model_selection import train_test_split
features = ["age", "bmi", "systolic_bp", "diastolic_bp", "heart_rate", "fasting_glucose", "hba1c",
            "total_cholesterol", "exercise_days_per_week", "family_history_flag",
            "medication_adherence_pct", "visits_last_year", "is_smoker", "is_male"]
data = df[df["risk_group"].notna()]
X, y = data[features].astype(float), data["risk_group"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
X_train.shape, X_test.shape

## 3. Train three classifiers (imputation inside the pipeline -> no test-set leakage)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
models = {
    "Logistic Regression": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42)),
    "Gradient Boosting": make_pipeline(SimpleImputer(strategy="median"), GradientBoostingClassifier(random_state=42)),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name:<20} train {model.score(X_train, y_train):.3f}  test {model.score(X_test, y_test):.3f}")

## 4. Evaluation — confusion matrix, precision, recall, F1

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
labels = ["Low", "Moderate", "High"]
pred = models["Gradient Boosting"].predict(X_test)
print(confusion_matrix(y_test, pred, labels=labels))
print(classification_report(y_test, pred, labels=labels))

## 5. Regression metrics (MAE, RMSE, R²) — R² is *not* accuracy

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
reg = data[data["risk_score"].notna()]
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(reg[features].astype(float), reg["risk_score"], test_size=.25, random_state=42)
lin = make_pipeline(SimpleImputer(strategy="median"), LinearRegression()).fit(Xr_tr, yr_tr)
p = lin.predict(Xr_te)
print("MAE", mean_absolute_error(yr_te, p), "RMSE", mean_squared_error(yr_te, p) ** .5, "R2", r2_score(yr_te, p))

## 6. Leakage demonstration — why ‘too good to be true’ results need scrutiny

In [ ]:
Xl = data[features + ["risk_score"]].astype(float)
a, b, c, d = train_test_split(Xl, y, test_size=.25, stratify=y, random_state=42)
leaky = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42)).fit(a, c)
print("test accuracy WITH leakage:", round(leaky.score(b, d), 3))